Notebook 2 — Análise Exploratória
02_exploratory_data_analysis.ipynb

Esse notebook é focado em:

estatística;
visualização;
insights;
comportamento da inadimplência.

Aqui você atua como Data Analyst.

Primeiro passo

Ler parquet:

df_eda = spark.read.parquet(
    "data/processed/home_credit_featured.parquet"
)
Temas que entram aqui
3. Análise Exploratória (EDA)
Tudo aqui.

Inclua:

distribuição de renda;
distribuição de crédito;
idade;
TARGET;
análise bivariada;
correlação;
impacto de categóricas;
boxplots;
histogramas.
10. Análise de Resultados e Insights
Parte analítica aqui.

Inclua:

perfil do inadimplente;
variáveis relevantes;
comportamento financeiro;
insights de negócio;
interpretações.
Resultado do Notebook 2

Você entrega:

Insights e entendimento do comportamento da inadimplência


In [2]:
# Iniciando sessão Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Studys") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession iniciada com sucesso")

SparkSession iniciada com sucesso


In [3]:
# Importando Bibliotecas
import sys, os

# Garante que o Spark use o mesmo Python que está rodando o notebook
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, count
import pandas as pd

print(f"Python em uso: {sys.executable}")

Python em uso: C:\Users\isabe\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [4]:
import pandas as pd

df_final = spark.read.csv("DataSets/home_credit_tratado.csv", header=True, inferSchema=True)

df_final.limit(5).toPandas()

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,faixa_renda,faixa_idade,bureau_total_creditos,bureau_total_atraso,bureau_media_credito,bureau_flag_atraso,prev_qtd_aplicacoes,prev_aprovados,prev_recusados,prev_taxa_aprovacao
0,100010,0,M,Y,Y,0.0,360000.0,1530000.0,42075.0,1530000.0,...,muito_alta,idoso,2,0.0,495000.00,0,1,1,0,1.0
1,100014,0,F,N,Y,1.0,112500.0,652500.0,21177.0,652500.0,...,media,jovem,8,0.0,341241.55,0,2,2,0,1.0
2,100021,0,F,N,Y,1.0,81000.0,270000.0,13500.0,270000.0,...,baixa,jovem,0,0.0,0.00,0,6,6,0,1.0
3,100062,0,M,Y,N,0.0,81000.0,675000.0,32472.0,675000.0,...,baixa,idoso,2,0.0,540813.06,0,10,3,4,0.3
4,100070,0,M,Y,Y,0.0,540000.0,1227901.5,46899.0,1129500.0,...,muito_alta,idoso,6,0.0,1459800.00,0,2,2,0,1.0


In [5]:
df_final.count()

307511

# Preparação dos Dados para Modelagem

In [6]:
from pyspark.ml.feature import (
    StringIndexer,   # transforma texto em número (ex: "M"→0, "F"→1)
    OneHotEncoder,   # transforma o número em vetor binário para o modelo não criar hierarquia falsa
    VectorAssembler, # junta colunas em um vetor (entrada do modelo)
    StandardScaler   # normaliza variáveis numéricas
)
from pyspark.ml import Pipeline


In [7]:
# Transformando variáveis categóricas em numéricas para poderem ser interpretadas pelo modelo

# Definindo variáveis categóricas
cat_cols = [
    "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "faixa_renda", "faixa_idade"
]

indexers = []
encoders = []

for col in cat_cols:
    # Transforma texto em índices numéricos
    indexer = StringIndexer(
        inputCol=col,
        outputCol=col + "_idx", # para indicar que antes era numérico
        handleInvalid="keep"
    )

    # Transforma os índices em vetores binários para não ter hierarquia.
    # ex: M = [0, 1], F = [1, 0]
    encoder = OneHotEncoder(
        inputCol=col + "_idx",
        outputCol=col + "_ohe"
    )

    indexers.append(indexer)
    encoders.append(encoder)


# print("Colunas categóricas que serão transformadas:")
# for c in cat_cols:
#     print(f"  {c}  →  {c}_idx  →  {c}_ohe")

In [10]:
from pyspark.sql.functions import col
# Definindo variáveis numéricas
num_cols = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "CNT_CHILDREN", "credit_income_ratio", "annuity_income_ratio",
    "idade_anos", "anos_empregado",
    "bureau_total_creditos", "bureau_total_atraso", "bureau_media_credito", "bureau_flag_atraso",
    "prev_qtd_aplicacoes", "prev_aprovados", "prev_recusados", "prev_taxa_aprovacao"
]

ohe_cols = []

for c in cat_cols:
    ohe_cols.append(c + "_ohe")

assembler = VectorAssembler(
    # Junta todas as colunas (num e cat) em um único vetor chamado features
    inputCols=num_cols + ohe_cols,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    # Padroniza os valores numéricos
    inputCol="features_raw",
    outputCol="features",
    withStd=True,   # divide pelo desvio padrão
    withMean=False  # não centraliza na média
)

# O Spark ML usa o nome "label" como variável alvo padrão
df_model = df_final.withColumn(
    "label",
    col("TARGET").cast("int")
)

In [11]:
# Dividindo a base em 80% para treino e 20% para testes
df_train, df_test = df_model.randomSplit(
    [0.8, 0.2],
    seed=42 # serve para garantir que toda a execução gere uma nova a mesma divisão de dados
)
print(f"Treino: {df_train.count()} registros")
print(f"Teste : {df_test.count()} registros")

Treino: 246274 registros
Teste : 61237 registros


In [12]:
# Tratamento do desbalanceamento por undersampling (quando uma classe aparece muito mais do que a outra).
# Sem balanceamento, o modelo aprende algo enviesado, tipo: “Se eu sempre prever 0 (adimplente), acerto quase tudo”

# Quantidade de registros para cada classificação
qtdd_inadimplentes = df_train.filter(
    col("label") == 1
).count()

qtdd_adimplentes = df_train.filter(
    col("label") == 0
).count()

proporcao = qtdd_inadimplentes / qtdd_adimplentes # Proporção para fazer undersampling

print("Inadimplentes (1):", qtdd_inadimplentes)
print("Adimplentes (0):", qtdd_adimplentes)
print("Proporção:", proporcao)

# inadimplentes
df_inadimplentes = df_train.filter(
    col("label") == 1
)

# adimplentes
df_adimplentes = df_train.filter(
    col("label") == 0
).sample( #reduz a quantidade de registros
    fraction=proporcao,
    seed=42
)


df_train_bal = df_inadimplentes.union(
    df_adimplentes
)
df_train_bal = df_train_bal.cache() # Mantém os dados em memória evita recalcular durante o treino
df_train_bal.count()

print(
    "\nTotal registros balanceados:",
    df_train_bal.count()
)

df_train_bal.groupBy("label").count().show()


Inadimplentes (1): 19816
Adimplentes (0): 226458
Proporção: 0.08750408464262689

Total registros balanceados: 39747
+-----+-----+
|label|count|
+-----+-----+
|    1|19816|
|    0|19931|
+-----+-----+



## Sobre Undersampling
### O que é
É o balanceamento da quantidade de vezes que aparece um dado, de forma que remove os dados da classe que mais aparece para que se iguale a quantidade da classe que aparece menos vezes.
### Vantagens do Undersampling
Simples e rápido, reduz custo computacional, melhora aprendizado da classe minoritária.
### Desvantagens
Joga fora dados reais, perder informação importante, reduzir capacidade de generalização.
### Alternativas que você deve conhecer
- Oversampling: duplica ou gera dados da classe minoritária, ex: SMOTE.
- Class weights: penaliza mais erros na classe rara, muito usado em produção
- Ajuste de threshold: muda o corte de decisão do modelo
### Quando usar undersampling
- Use quando:
dataset é muito grande, classe majoritária domina absurdamente, exije rapidez
- Evite quando:
possui poucos dados, cada observação é valiosa

# Modelagem

In [13]:
# Treinando os modelos usando Pipeline do Spark ML
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

# Explicação dos parâmetros:
# - featuresCol="features" -> Indica qual coluna contém as variáveis de entrada.
# labelCol="label" -> Indica a variável alvo da análise para previsão
# maxDepth=5 -> Limita profundidade da árvore.


# Regressão Logística: modelo linear
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Árvore de Decisão: série de perguntas até chegar à decisão
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, seed=42)

# Random Forest: 20 árvores (reduzido de 50 para não derrubar a memória)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=5, seed=42)

def build_pipeline(modelo):
    return Pipeline(stages=indexers + encoders + [assembler, scaler, modelo])

print("Treinando Regressão Logística...")
pipeline_lr = build_pipeline(lr).fit(df_train_bal) # fit serve para treinar os modelos com base no df dividido que separamos para treino
print("OK — Treinando Árvore de Decisão...")
pipeline_dt = build_pipeline(dt).fit(df_train_bal)
print("OK — Treinando Random Forest (pode demorar 2-3 min)...")
pipeline_rf = build_pipeline(rf).fit(df_train_bal)
print("\nTodos os modelos treinados com sucesso!")

Treinando Regressão Logística...
OK — Treinando Árvore de Decisão...
OK — Treinando Random Forest (pode demorar 2-3 min)...

Todos os modelos treinados com sucesso!


# Explicação sobre os modelos de Machine Learning
## Regressão Logística (LogisticRegression)
Tipo: modelo linear
Como funciona: combina as variáveis com pesos para tentar encontrar a probabilidade de classe (0 ou 1)
Uso típico: baseline e problemas onde relações são mais simples

Prós: rápido, interpretável
Contras: não captura bem relações complexas

## Árvore de Decisão (DecisionTree)
Tipo: modelo não linear baseado em regras
Como funciona: cria divisões sequenciais (“se… então…”) nos dados, ex: renda > 5000? sim -> ...; não -> ...

Prós: fácil de interpretar, captura padrões complexos
Contras: tende a overfitting (por isso limitar profundidade)

## Random Forest (RandomForest)
Tipo: ensemble (conjunto de árvores)
Como funciona: treina várias árvores e combina os resultados (votação)

Prós: mais robusto, melhor desempenho geral, reduz overfitting (ocorre quando o modelo aprende excessivamente os dados de treinamento, memorizando ruídos e detalhes específicos em vez de aprender os padrões gerais)
Contras: mais pesado e menos interpretável

In [40]:
# Avaliando os resultados do treino dos modelos
from pyspark.ml.evaluation import (BinaryClassificationEvaluator # usado para classificação binária (ex.: caloteiro ou não caloteiro)
, MulticlassClassificationEvaluator # calcula métricas como acurácia, precisão, recall e F1
)

def avaliar_modelo(nome, pipeline, df_teste):
    preds = pipeline.transform(df_teste)
    ev   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    # ROC-AUC serve para medir o quanto o modelo consegue separar corretamente as classes (0.5=chute, 1.0=perfeito)
    auc  = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds)
    # ACC acurácia é o percentual total de acertos.
    acc  = ev.evaluate(preds, {ev.metricName: "accuracy"})
    # PREC - Precisão dos clientes previstos como “inadimplentes”, quantos realmente eram.
    prec = ev.evaluate(preds, {ev.metricName: "weightedPrecision"})
    # REC - Recall dos clientes inadimplentes reais, quantos o modelo conseguiu encontrar.
    rec  = ev.evaluate(preds, {ev.metricName: "weightedRecall"})
    # F1 - Média equilibrada entre Precisão e Recall.
    f1   = ev.evaluate(preds, {ev.metricName: "f1"})

    return {"Modelo": nome, "ROC-AUC": round(auc,4), "Acurácia": round(acc,4), "Precisão": round(prec,4),
            "Recall": round(rec,4), "F1": round(f1,4)}

resultados = []
resultados.append(avaliar_modelo("Regressão Logística", pipeline_lr, df_test))
resultados.append(avaliar_modelo("Árvore de Decisão",   pipeline_dt, df_test))
resultados.append(avaliar_modelo("Random Forest",        pipeline_rf, df_test))
df_resultados = pd.DataFrame(resultados).set_index("Modelo")
print(f"{' Comparação dos Modelos ':=^70}")
print(df_resultados.to_string())


# O ROC-AUC é a melhor variável para determinar a capacidade do modelo, porque ela ordena clientes do mais ao mesmo arriscado de inadimplência.
melhor_opc = df_resultados["ROC-AUC"].idxmax()
print(f"\nMelhor modelo (maior ROC-AUC): {melhor_opc}")


======================= Comparação dos Modelos =======================
                     ROC-AUC  Acurácia  Precisão  Recall      F1
Modelo                                                          
Regressão Logística   0.6707    0.6263    0.8821  0.6263  0.7106
Árvore de Decisão     0.5385    0.6286    0.8766  0.6286  0.7125
Random Forest         0.6539    0.6199    0.8793  0.6199  0.7056

Melhor modelo (maior ROC-AUC): Regressão Logística


In [43]:
# Testando como a Regressáo Logística se sai com a base de dados de teste.

melhor_mod = df_resultados["ROC-AUC"].idxmax()
melhor_auc  = df_resultados.loc[melhor_mod, "ROC-AUC"]

melhor_pipeline = {
    "Regressão Logística": pipeline_lr,
    "Árvore de Decisão":   pipeline_dt,
    "Random Forest":       pipeline_rf
}[melhor_mod]

print(f"Melhor modelo selecionado : {melhor_mod}")
print(f"ROC-AUC                   : {melhor_auc}")

previsoes = melhor_pipeline.transform(df_test)

print("Previsões geradas com sucesso!")
# label = real (0 = adimplente, 1 = inadimplente)
# previão = o que o modelo previu
# probabilidade (0,1)
previsoes.select("SK_ID_CURR", "label", "prediction", "probability").limit(5).toPandas()

Melhor modelo selecionado : Regressão Logística
ROC-AUC                   : 0.6707
Previsões geradas com sucesso!


,SK_ID_CURR,label,prediction,probability
0,100021,0,1.0,"[0.487819666677544, 0.5121803333224559]"
1,100062,0,1.0,"[0.44752907766409794, 0.5524709223359021]"
2,100087,0,0.0,"[0.5429808743899958, 0.45701912561000424]"
3,100129,0,0.0,"[0.8536053463746903, 0.14639465362530968]"
4,100157,0,1.0,"[0.4577186796982808, 0.5422813203017192]"


In [52]:
# Salvando o df em csv para utilização em outros notebooks
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array

# transformando a coluna probability em uma coluna simples
previsoes_export = (
    previsoes
    .withColumn(
        "prob_adimplente",
        vector_to_array(col("probability"))[0]
    )
    .withColumn(
        "prob_inadimplente",
        vector_to_array(col("probability"))[1]
    )
    .drop("probability")
)

df_pandas = previsoes_export.toPandas()

df_pandas.to_csv(
    "DataSets/previsoes.csv",
    index=False
)
